---
title: Piping in Python
subtitle: Chaining Data Transformations with Python's dfply
abstract: Piping is a programming paradigm that allows chaining multiple operations together in a readable and concise manner. It is inspired by the pipe operator (%>%) in R's dplyr package and is implemented in Python using libraries like dfply. This approach is particularly useful for data manipulation tasks, as it enables a clean and logical flow of operations on data.
author:
  - name: Karol Flisikowski
    affiliations: 
      - Gdansk University of Technology
      - Chongqing Technology and Business University
    orcid: 0000-0002-4160-1297
    linkedin: flisik
    email: karol@ctbu.edu.cn
date: 2025-04-28
---

## Piping in Python

Learn how to summarize the columns available in an R data frame. You will also learn how to chain operations together with the pipe operator, and how to compute grouped summaries using.

The dfply package makes it possible to do R's dplyr-style data manipulation with pipes in python on pandas DataFrames.

[dfply website here](https://github.com/kieferk/dfply)

[![](https://www.rforecology.com/pipes_image0.png "https://github.com/kieferk/dfply")](https://github.com/kieferk/dfply)

**Key Features of Piping:**

- Chaining Operations: Use the >> operator to chain multiple operations on a pandas DataFrame.
- Deferred Evaluation: Operations are recorded symbolically and evaluated only when needed.
- Readable Syntax: Makes complex data transformations easier to read and understand.

**Common Functions in dfply:**

- select(): Select specific columns.
- drop(): Drop specific columns.
- filter(): Filter rows based on conditions.
- mutate(): Add or modify columns.
- summarize(): Compute summary statistics.
- group_by(): Group data for aggregation.

Piping simplifies workflows by reducing the need for intermediate variables and making the code more intuitive

In [1]:
%pip install seaborn dfply
import pandas as pd
import seaborn as sns
cars = sns.load_dataset('mpg')
from dfply import *
cars >> head(3)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite


## The \>\> and \>\>=

dfply works directly on pandas DataFrames, chaining operations on the data with the >> operator, or alternatively starting with >>= for inplace operations.

*The X DataFrame symbol*

The DataFrame as it is passed through the piping operations is represented by the symbol X. It records the actions you want to take (represented by the Intention class), but does not evaluate them until the appropriate time. Operations on the DataFrame are deferred. Selecting two of the columns, for example, can be done using the symbolic X DataFrame during the piping operations.

### Exercise 1.

Select the columns 'mpg' and 'horsepower' from the cars DataFrame.

In [5]:
cars >> select('mpg','horsepower') >> head(5) 

,mpg,horsepower
0,18.0,130.0
1,15.0,165.0
2,18.0,150.0
3,16.0,150.0
4,17.0,140.0


## Selecting and dropping

There are two functions for selection, inverse of each other: select and drop. The select and drop functions accept string labels, integer positions, and/or symbolically represented column names (X.column). They also accept symbolic "selection filter" functions, which will be covered shortly.

### Exercise 2.

Select the columns 'mpg' and 'horsepower' from the cars DataFrame using the drop function.

In [ ]:
# your solution goes here
cars >> drop(~X.mpg, ~X.horsepower) >> head(5)

#cars.drop(columns=[~X.mpg, ~X.horsepower]) >> head(5) nie działa 

#columns_to_drop = cars.columns.difference(['mpg', 'horsepower'])

#cars.drop(columns=columns_to_drop).head(5)

,mpg,horsepower
0,18.0,130.0
1,15.0,165.0
2,18.0,150.0
3,16.0,150.0
4,17.0,140.0


## Selection using \~

One particularly nice thing about dplyr's selection functions is that you can drop columns inside of a select statement by putting a subtraction sign in front, like so: ... %>% select(-col). The same can be done in dfply, but instead of the subtraction operator you use the tilde ~.

### Exercise 3.

Select all columns except 'model_year', and 'name' from the cars DataFrame.

In [8]:
# your solution goes here

cars>> select(~X.model_year, ~X.name) >> head(5)

,mpg,cylinders,displacement,horsepower,weight,acceleration,origin
0,18.0,8,307.0,130.0,3504,12.0,usa
1,15.0,8,350.0,165.0,3693,11.5,usa
2,18.0,8,318.0,150.0,3436,11.0,usa
3,16.0,8,304.0,150.0,3433,12.0,usa
4,17.0,8,302.0,140.0,3449,10.5,usa


## Filtering columns

The vanilla select and drop functions are useful, but there are a variety of selection functions inspired by dplyr available to make selecting and dropping columns a breeze. These functions are intended to be put inside of the select and drop functions, and can be paired with the ~ inverter.

First, a quick rundown of the available functions:

-   starts_with(prefix): find columns that start with a string prefix.
-   ends_with(suffix): find columns that end with a string suffix.
-   contains(substr): find columns that contain a substring in their name.
-   everything(): all columns.
-   columns_between(start_col, end_col, inclusive=True): find columns between a specified start and end column. The inclusive boolean keyword argument indicates whether the end column should be included or not.
-   columns_to(end_col, inclusive=True): get columns up to a specified end column. The inclusive argument indicates whether the ending column should be included or not.
-   columns_from(start_col): get the columns starting at a specified column.

### Exercise 4.

The selection filter functions are best explained by example. Let's say I wanted to select only the columns that started with a "c":

In [ ]:
# your solution goes here
cars >>select(starts_with('c')) >>head(5)

""
0
1
2
3
4


### Exercise 5.

Select the columns that contain the substring "e" from the cars DataFrame.

In [17]:
# your solution goes here
cars >>select(~starts_with('c')) >>head(5)

,mpg,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,302.0,140.0,3449,10.5,70,usa,ford torino


### Exercise 6.

Select the columns that are between 'mpg' and 'origin' from the cars DataFrame.

In [23]:
# your solution goes here

cars >> select(columns_between(X.mpg, X.origin)) >> head(5)

#cars.iloc[:, [1, 2, 6]]

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504,12.0,70,usa
1,15.0,8,350.0,165.0,3693,11.5,70,usa
2,18.0,8,318.0,150.0,3436,11.0,70,usa
3,16.0,8,304.0,150.0,3433,12.0,70,usa
4,17.0,8,302.0,140.0,3449,10.5,70,usa


## Subsetting and filtering

### row_slice()

Slices of rows can be selected with the row_slice() function. You can pass single integer indices or a list of indices to select rows as with. This is going to be the same as using pandas' .iloc.

#### Exercise 7.

Select the first three rows from the cars DataFrame.

In [24]:
# your solution goes here

cars >> row_slice([0,1,3]) >> head(5)


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst


### distinct()

Selection of unique rows is done with distinct(), which similarly passes arguments and keyword arguments through to the DataFrame's .drop_duplicates() method.

#### Exercise 8.

Select the unique rows from the 'origin' column in the cars DataFrame.

In [27]:
# your solution goes here

cars>>select('origin')>>distinct('origin') 

,origin
0,usa
14,japan
19,europe


## mask()

Filtering rows with logical criteria is done with mask(), which accepts boolean arrays "masking out" False labeled rows and keeping True labeled rows. These are best created with logical statements on symbolic Series objects as shown below. Multiple criteria can be supplied as arguments and their intersection will be used as the mask.

### Exercise 9.

Filter the cars DataFrame to only include rows where the 'mpg' is greater than 20, origin Japan, and display the first three rows:

In [2]:
# your solution goes here

cars >>mask(X.origin=='japan', X.mpg>20) >> head(5)



,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
14,24.0,4,113.0,95.0,2372,15.0,70,japan,toyota corona mark ii
18,27.0,4,97.0,88.0,2130,14.5,70,japan,datsun pl510
29,27.0,4,97.0,88.0,2130,14.5,71,japan,datsun pl510
31,25.0,4,113.0,95.0,2228,14.0,71,japan,toyota corona
53,31.0,4,71.0,65.0,1773,19.0,71,japan,toyota corolla 1200


## pull()

The pull() function is used to extract a single column from a DataFrame as a pandas Series. This is useful for passing a single column to a function or for further manipulation.

### Exercise 10.

Extract the 'mpg' column from the cars DataFrame, japanese origin, model year 70s, and display the first three rows.

In [12]:
result = (cars
          .query("origin == 'japan' and model_year == 70")
          .mpg
          .head(3))

## DataFrame transformation

*mutate()*

The mutate() function is used to create new columns or modify existing columns. It accepts keyword arguments of the form new_column_name = new_column_value, where new_column_value is a symbolic Series object.

### Exercise 11.

Create a new column 'mpg_per_cylinder' in the cars DataFrame that is the result of dividing the 'mpg' column by the 'cylinders' column.

In [11]:
# your solution goes here

cars >> mutate(mpg_per_cylinder = X.mpg / X.cylinders) >> head(5)

##cars["mpg_per_cylinder"] = cars["mpg"] / cars["cylinders"]

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name,mpg_per_cylinder
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu,2.250
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320,1.875
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite,2.250
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst,2.000
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino,2.125



*transmute()*

The transmute() function is a combination of a mutate and a selection of the created variables.

### Exercise 12.

Create a new column 'mpg_per_cylinder' in the cars DataFrame that is the result of dividing the 'mpg' column by the 'cylinders' column, and display only the new column.

In [20]:
# your solution goes here

cars >> transmute(mpg_per_cylinder = X.mpg / X.cylinders) 

,mpg_per_cylinder
0,2.250
1,1.875
2,2.250
3,2.000
4,2.125
...,...
393,6.750
394,11.000
395,8.000
396,7.000


## Grouping

*group_by() and ungroup()*

The group_by() function is used to group the DataFrame by one or more columns. This is useful for creating groups of rows that can be summarized or transformed together. The ungroup() function is used to remove the grouping.

### Exercise 13.

Group the cars DataFrame by the 'origin' column and calculate the lead of the 'mpg' column.

In [35]:
# your solution goes here
%pip install pandas

##cars >> group_by(X.origin) >> arrange(X.mpg, ascending=False) >> head(5)

cars.sort_values('mpg', ascending=False).groupby('origin').head(5)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
322,46.6,4,86.0,65.0,2110,17.9,80,japan,mazda glc
329,44.6,4,91.0,67.0,1850,13.8,80,japan,honda civic 1500 gl
325,44.3,4,90.0,48.0,2085,21.7,80,europe,vw rabbit c (diesel)
394,44.0,4,97.0,52.0,2130,24.6,82,europe,vw pickup
326,43.4,4,90.0,48.0,2335,23.7,80,europe,vw dasher (diesel)
244,43.1,4,90.0,48.0,1985,21.5,78,europe,volkswagen rabbit custom diesel
309,41.5,4,98.0,76.0,2144,14.7,80,europe,vw rabbit
324,40.8,4,85.0,65.0,2110,19.2,80,japan,datsun 210
247,39.4,4,85.0,70.0,2070,18.6,78,japan,datsun b210 gx
343,39.1,4,79.0,58.0,1755,16.9,81,japan,toyota starlet


## Reshaping

*arrange()*

The arrange() function is used to sort the DataFrame by one or more columns. This is useful for reordering the rows of the DataFrame.

### Exercise 14.

Sort the cars DataFrame by the 'mpg' column in descending order.

In [32]:
# your solution goes here

#cars >> arrange(X.mpg, ascending=False) >> head(5)

cars.sort_values('mpg', ascending=False).head(5)

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
322,46.6,4,86.0,65.0,2110,17.9,80,japan,mazda glc
329,44.6,4,91.0,67.0,1850,13.8,80,japan,honda civic 1500 gl
325,44.3,4,90.0,48.0,2085,21.7,80,europe,vw rabbit c (diesel)
394,44.0,4,97.0,52.0,2130,24.6,82,europe,vw pickup
326,43.4,4,90.0,48.0,2335,23.7,80,europe,vw dasher (diesel)



*rename()*

The rename() function is used to rename columns in the DataFrame. It accepts keyword arguments of the form new_column_name = old_column_name.

### Exercise 15.

Rename the 'mpg' column to 'miles_per_gallon' in the cars DataFrame.

In [36]:
# your solution goes here

cars >>rename(miles_per_gallon = X.mpg)

#cars.rename(columns={'mpg':'miles_per_gallon'})

,miles_per_gallon,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino
...,...,...,...,...,...,...,...,...,...
393,27.0,4,140.0,86.0,2790,15.6,82,usa,ford mustang gl
394,44.0,4,97.0,52.0,2130,24.6,82,europe,vw pickup
395,32.0,4,135.0,84.0,2295,11.6,82,usa,dodge rampage
396,28.0,4,120.0,79.0,2625,18.6,82,usa,ford ranger



*gather()*

The gather() function is used to reshape the DataFrame from wide to long format. It accepts keyword arguments of the form new_column_name = new_column_value, where new_column_value is a symbolic Series object.

### Exercise 16.

Reshape the cars DataFrame from wide to long format by gathering the columns 'mpg', 'horsepower', 'weight', 'acceleration', and 'displacement' into a new column 'variable' and their values into a new column 'value'.

In [ ]:
# your solution goes here


*spread()*

Likewise, you can transform a "long" DataFrame into a "wide" format with the spread(key, values) function. Converting the previously created elongated DataFrame for example would be done like so.

### Exercise 17.

Reshape the cars DataFrame from long to wide format by spreading the 'variable' column into columns and their values into the 'value' column.

In [ ]:
# your solution goes here


## Summarization

*summarize()*

The summarize() function is used to calculate summary statistics for groups of rows. It accepts keyword arguments of the form new_column_name = new_column_value, where new_column_value is a symbolic Series object.

### Exercise 18.

Calculate the mean 'mpg' for each group of 'origin' in the cars DataFrame.

In [ ]:
# your solution goes here


*summarize_each()*

The summarize_each() function is used to calculate summary statistics for groups of rows. It accepts keyword arguments of the form new_column_name = new_column_value, where new_column_value is a symbolic Series object.

### Exercise 19.

Calculate the mean 'mpg' and 'horsepower' for each group of 'origin' in the cars DataFrame.

In [ ]:
# your solution goes here


*summarize() can of course be used with groupings as well.*

### Exercise 20.

Calculate the mean 'mpg' for each group of 'origin' and 'model_year' in the cars DataFrame.

In [ ]:
# your solution goes here